# Tugas 1 — Pengantar Pemrosesan Paralel

Nama :Najwa Azzahra                                       
NIM : 2411513004                                     
Mata Kuliah: Pemrosesan Paralel dengan Python                                                
Topik:Flynn, Hukum Amdahl, Benchmarking, dan Hukum Gustafson

## 1. Klasifikasi Flynn

Klasifikasi Flynn ditentukan dari jumlah aliran instruksi dan aliran data yang diproses secara bersamaan.

**1. `arr + arr` pada NumPy array**  
**Kategori: SIMD**  
Satu operasi/instruksi yang sama diterapkan secara serentak pada banyak elemen data.

**2. Program Python biasa tanpa threading/multiprocessing yang membaca file lalu memproses baris demi baris**  
**Kategori: SISD**  
Instruksi diproses secara sekuensial terhadap data satu per satu.

**3. Cluster komputer dengan setiap node menjalankan simulasi cuaca berbeda untuk wilayah berbeda secara independen**  
**Kategori: MIMD**  
Banyak node dapat menjalankan instruksi/program berbeda terhadap data yang berbeda secara independen.

**Kesimpulan:** ketiga skenario tersebut masing-masing mewakili SIMD, SISD, dan MIMD. Tidak ada skenario MISD pada soal ini.

## 2. Analisis Hukum Amdahl

Diketahui bagian program yang dapat diparalelkan:

$$P = 0.85$$

Rumus Hukum Amdahl:

$$S(N)=\frac{1}{(1-P)+P/N}$$

Perhitungan dilakukan untuk jumlah prosesor `N = 2, 4, 8, 16, 32, 64`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def amdahl_speedup(P, N):
    return 1 / ((1 - P) + P / N)

P = 0.85
N_values = np.array([2, 4, 8, 16, 32, 64])

speedups = [amdahl_speedup(P, N) for N in N_values]

for N, S in zip(N_values, speedups):
    print(f"N = {N:>2} prosesor -> Speedup teoritis = {S:.3f}x")

### Hasil

Dari perhitungan diperoleh:

- `N = 2` → Speedup = **1,739x**
- `N = 4` → Speedup = **2,759x**
- `N = 8` → Speedup = **3,902x**
- `N = 16` → Speedup = **4,706x**
- `N = 32` → Speedup = **5,081x**
- `N = 64` → Speedup = **5,340x**

Speedup maksimum ketika jumlah prosesor mendekati tak hingga adalah:

$$S_{max}=\frac{1}{1-P}=\frac{1}{0.15}=6.667$$

Artinya, walaupun jumlah prosesor terus ditambah, speedup tidak dapat melampaui sekitar **6,667x** karena masih terdapat 15% bagian serial.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(N_values, speedups, marker="o")
plt.axhline(1/(1-P), linestyle="--", label="Batas maksimum Amdahl")
plt.title("Hukum Amdahl — P = 0.85")
plt.xlabel("Jumlah Prosesor (N)")
plt.ylabel("Speedup")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Diminishing returns

Diminishing returns mulai terlihat jelas sekitar **N = 16–32 prosesor**. Setelah itu, penambahan prosesor hanya memberikan kenaikan speedup yang semakin kecil.

Contohnya:

- 16 → 32 prosesor: speedup naik sekitar **0,375x**.
- 32 → 64 prosesor: speedup hanya naik sekitar **0,259x**.

Penyebabnya adalah bagian serial sebesar 15% tetap harus dijalankan secara berurutan. Ketika jumlah prosesor semakin besar, waktu yang dapat dipercepat semakin kecil dibandingkan bagian serial tersebut.

## 3. Benchmark Mandiri

Untuk benchmark digunakan fungsi CPU-bound buatan sendiri yang menghitung jumlah kuadrat dari `1` sampai `N`.

Dua metode pengukuran digunakan:

1. `time.perf_counter()` untuk mengukur satu eksekusi.
2. `timeit` untuk melakukan pengukuran berulang sehingga hasil lebih stabil.

Empat ukuran input digunakan: **100.000, 200.000, 400.000, dan 800.000**.

In [ ]:
import time
import timeit

def jumlah_kuadrat(n):
    total = 0
    for i in range(1, n + 1):
        total += i * i
    return total

ukuran_input = [100_000, 200_000, 400_000, 800_000]
hasil_benchmark = []

for n in ukuran_input:
    start = time.perf_counter()
    hasil = jumlah_kuadrat(n)
    waktu_perf = time.perf_counter() - start

    waktu_timeit = timeit.repeat(
        stmt="jumlah_kuadrat(n)",
        globals={"jumlah_kuadrat": jumlah_kuadrat, "n": n},
        number=1,
        repeat=5
    )

    waktu_terbaik = min(waktu_timeit)
    waktu_rata_rata = sum(waktu_timeit) / len(waktu_timeit)

    hasil_benchmark.append((n, waktu_perf, waktu_terbaik, waktu_rata_rata))

    print(f"Input N = {n:,}")
    print(f"perf_counter     = {waktu_perf:.6f} detik")
    print(f"timeit terbaik   = {waktu_terbaik:.6f} detik")
    print(f"timeit rata-rata = {waktu_rata_rata:.6f} detik")
    print(f"Hasil perhitungan = {hasil:,}")
    print()

### Penjelasan hasil benchmark

Nilai `perf_counter()` dapat berbeda dengan hasil `timeit` karena `perf_counter()` hanya mengukur satu eksekusi dan dapat dipengaruhi aktivitas sistem pada saat pengukuran.

Sebaliknya, `timeit` melakukan beberapa pengulangan sehingga dapat memberikan gambaran yang lebih stabil. Karena fungsi yang diuji bersifat CPU-bound, waktu eksekusi juga meningkat ketika ukuran `N` diperbesar.

In [ ]:
input_plot = [data[0] for data in hasil_benchmark]
perf_plot = [data[1] for data in hasil_benchmark]
timeit_avg_plot = [data[3] for data in hasil_benchmark]

plt.figure(figsize=(8, 5))
plt.plot(input_plot, perf_plot, marker="o", label="time.perf_counter()")
plt.plot(input_plot, timeit_avg_plot, marker="s", label="timeit rata-rata")
plt.title("Benchmark Fungsi CPU-bound")
plt.xlabel("Ukuran Input (N)")
plt.ylabel("Waktu Eksekusi (detik)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 4. Refleksi Singkat

**Hukum Amdahl** lebih relevan ketika ukuran masalah relatif tetap dan tujuan utama adalah mengetahui seberapa cepat masalah yang sama dapat diselesaikan dengan menambah prosesor. Contohnya adalah mempercepat proses pengolahan satu dataset citra dengan jumlah core yang lebih banyak. Hukum Amdahl menunjukkan bahwa bagian serial menjadi batas speedup.

Sebaliknya, **Hukum Gustafson** lebih relevan ketika penambahan prosesor diikuti dengan peningkatan ukuran masalah. Contohnya adalah simulasi cuaca pada sistem HPC: ketika jumlah node bertambah, resolusi wilayah, jumlah variabel, atau durasi simulasi dapat diperbesar sehingga sumber daya tambahan tetap dimanfaatkan. Karena itu, Gustafson lebih sesuai untuk menggambarkan *scaled speedup* pada masalah yang ukurannya berkembang.

## Kesimpulan

1. `arr + arr` pada NumPy dikategorikan **SIMD**, program Python sekuensial dikategorikan **SISD**, dan cluster dengan pekerjaan berbeda pada tiap node dikategorikan **MIMD**.
2. Untuk `P = 0.85`, speedup Amdahl mendekati batas maksimum **6,667x** dan mulai mengalami diminishing returns ketika jumlah prosesor semakin besar, terutama sekitar 16–32 prosesor ke atas.
3. `time.perf_counter()` cocok untuk pengukuran langsung, sedangkan `timeit` lebih stabil karena melakukan pengulangan benchmark.
4. Amdahl cocok untuk *strong scaling* dengan ukuran masalah tetap, sedangkan Gustafson cocok untuk *weak scaling* ketika ukuran masalah bertambah bersama jumlah prosesor.